In [ ]:
!pip install optuna xgboost

In [ ]:
# -*- coding: utf-8 -*-
"""
타이타닉 생존 예측 - 리팩토링 버전
  1) 데이터 로드 및 (상태 없는) 정제
  2) 전처리 Pipeline (결측치/인코딩/스케일링을 fold 안에서 fit -> 데이터 누수 방지)
  3) 6개 모델 baseline 5-Fold 평가
  4) Optuna로 6개 모델 전부 하이퍼파라미터 튜닝 -> best 값 확인

Colab 사전 설치:  !pip install optuna xgboost
"""

import warnings
from typing import Callable

import numpy as np
import pandas as pd
import optuna
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------------------
# 설정
# ---------------------------------------------------------------------
TRAIN_URL = (
    "https://raw.githubusercontent.com/wesm/pydata-book/"
    "3rd-edition/datasets/titanic/train.csv"
)
RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 50            # 모델별 Optuna trial 수
TUNING_METRIC = "f1"     # 'f1', 'roc_auc', 'average_precision' 등으로 변경 가능

TITLE_MAPPING = {
    "Mr": "Mr", "Miss": "Miss", "Mrs": "Mrs", "Master": "Master",
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    **{t: "Rare" for t in [
        "Lady", "Countess", "Capt", "Col", "Don", "Dr",
        "Major", "Rev", "Sir", "Jonkheer", "Dona",
    ]},
}

NUM_COLS = ["Age", "Fare", "SibSp", "Parch"]
CAT_COLS = ["Sex", "Embarked", "Title"]
PASS_COLS = ["Pclass"]


# ---------------------------------------------------------------------
# 1. 데이터 로드 및 정제 (통계량을 쓰지 않는 변환만 수행)
# ---------------------------------------------------------------------
def load_and_clean(url: str = TRAIN_URL) -> tuple[pd.DataFrame, pd.Series]:
    df = pd.read_csv(url)
    df["Title"] = (
        df["Name"]
        .str.extract(r" ([A-Za-z]+)\.", expand=False)
        .map(TITLE_MAPPING)
        .fillna("Rare")
    )
    df = df.drop(columns=["PassengerId", "Ticket", "Cabin", "Name"])
    return df.drop(columns="Survived"), df["Survived"]


def summarize(df: pd.DataFrame, n_samples: int = 5) -> pd.DataFrame:
    """컬럼별 dtype, 고유값 수, 결측치 수/비율, 샘플 값을 요약."""
    return pd.DataFrame({
        "Column": df.columns,
        "Dtype": df.dtypes.values,
        "N_Unique": df.nunique().values,
        "Null_Count": df.isnull().sum().values,
        "Null_Ratio(%)": (df.isnull().mean() * 100).round(2).values,
        "Samples": [list(df[c].dropna().unique()[:n_samples]) for c in df.columns],
    })


# ---------------------------------------------------------------------
# 2. 전처리 Pipeline
# ---------------------------------------------------------------------
class GroupMedianImputer(BaseEstimator, TransformerMixin):
    """그룹(Pclass, Title)별 중앙값으로 target 결측치를 대체. fit 데이터 기준으로만 계산."""

    def __init__(self, target: str = "Age", groups: tuple = ("Pclass", "Title")):
        self.target = target
        self.groups = groups

    def fit(self, X: pd.DataFrame, y=None):
        self.group_medians_ = X.groupby(list(self.groups))[self.target].median()
        self.global_median_ = X[self.target].median()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        keys = pd.MultiIndex.from_frame(X[list(self.groups)])
        fill = pd.Series(self.group_medians_.reindex(keys).values, index=X.index)
        X[self.target] = X[self.target].fillna(fill).fillna(self.global_median_)
        return X


def build_preprocessor() -> Pipeline:
    column_tf = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), NUM_COLS),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),   # Embarked -> 최빈값
            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ]), CAT_COLS),
        ("pass", "passthrough", PASS_COLS),
    ])
    return Pipeline([
        ("age_impute", GroupMedianImputer("Age", ("Pclass", "Title"))),
        ("column_tf", column_tf),
    ])


def build_pipeline(model: BaseEstimator) -> Pipeline:
    return Pipeline([("prep", build_preprocessor()), ("model", model)])


# ---------------------------------------------------------------------
# 3. 모델 레지스트리 (모델 클래스, 고정 파라미터, Optuna 탐색 공간)
# ---------------------------------------------------------------------
def _knn(t: optuna.Trial) -> dict:
    return {
        "n_neighbors": t.suggest_int("n_neighbors", 3, 31),
        "weights": t.suggest_categorical("weights", ["uniform", "distance"]),
        "p": t.suggest_int("p", 1, 2),
    }


def _logreg(t: optuna.Trial) -> dict:
    return {
        "C": t.suggest_float("C", 1e-3, 1e2, log=True),
        "penalty": t.suggest_categorical("penalty", ["l1", "l2"]),
    }


def _nb(t: optuna.Trial) -> dict:
    return {"var_smoothing": t.suggest_float("var_smoothing", 1e-11, 1e-5, log=True)}


def _tree(t: optuna.Trial) -> dict:
    return {
        "criterion": t.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth": t.suggest_int("max_depth", 2, 12),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
    }


def _rf(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 100, 400, step=50),
        "max_depth": t.suggest_int("max_depth", 3, 15),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
        "max_features": t.suggest_categorical("max_features", ["sqrt", "log2"]),
    }


def _xgb(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 50, 300, step=50),
        "max_depth": t.suggest_int("max_depth", 3, 8),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": t.suggest_int("min_child_weight", 1, 7),
        "gamma": t.suggest_float("gamma", 0.0, 0.5),
    }


# name -> (estimator class, fixed params, search-space function)
MODEL_REGISTRY: dict[str, tuple[type, dict, Callable]] = {
    "KNN": (KNeighborsClassifier, {}, _knn),
    "Logistic Regression": (
        LogisticRegression,
        {"solver": "liblinear", "max_iter": 1000, "random_state": RANDOM_STATE},
        _logreg,
    ),
    "Naive Bayes": (GaussianNB, {}, _nb),
    "Decision Tree": (DecisionTreeClassifier, {"random_state": RANDOM_STATE}, _tree),
    "Random Forest": (RandomForestClassifier, {"random_state": RANDOM_STATE}, _rf),
    "XGBoost": (
        XGBClassifier,
        {"random_state": RANDOM_STATE, "eval_metric": "logloss"},
        _xgb,
    ),
}


def make_model(name: str, params: dict | None = None) -> BaseEstimator:
    cls, fixed, _ = MODEL_REGISTRY[name]
    return cls(**fixed, **(params or {}))


# ---------------------------------------------------------------------
# 4. 평가 함수 (Stratified K-Fold)
# ---------------------------------------------------------------------
def evaluate_cv(pipeline: Pipeline, X: pd.DataFrame, y: pd.Series, cv) -> dict:
    metrics: dict[str, list] = {
        "FN+FP (오분류 수)": [], "FPR (오탐률)": [], "Recall (재현율)": [],
        "Precision (정밀도)": [], "F1-Score": [], "PR_AUC": [], "AUROC": [],
    }
    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model = clone(pipeline).fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
        metrics["FN+FP (오분류 수)"].append(fn + fp)
        metrics["FPR (오탐률)"].append(fp / (fp + tn) if (fp + tn) else 0)
        metrics["Recall (재현율)"].append(recall_score(y_te, y_pred, zero_division=0))
        metrics["Precision (정밀도)"].append(precision_score(y_te, y_pred, zero_division=0))
        metrics["F1-Score"].append(f1_score(y_te, y_pred, zero_division=0))
        metrics["PR_AUC"].append(average_precision_score(y_te, y_prob))
        metrics["AUROC"].append(roc_auc_score(y_te, y_prob))

    return {k: float(np.mean(v)) for k, v in metrics.items()}


def evaluate_models(models: dict[str, BaseEstimator], X, y, cv) -> pd.DataFrame:
    rows = [
        {"Model": name, **evaluate_cv(build_pipeline(model), X, y, cv)}
        for name, model in models.items()
    ]
    return pd.DataFrame(rows).round(4)


# ---------------------------------------------------------------------
# 5. Optuna 하이퍼파라미터 튜닝
# ---------------------------------------------------------------------
def tune_model(name: str, X, y, cv, n_trials: int = N_TRIALS,
               scoring: str = TUNING_METRIC) -> optuna.Study:
    _, _, space_fn = MODEL_REGISTRY[name]

    def objective(trial: optuna.Trial) -> float:
        pipeline = build_pipeline(make_model(name, space_fn(trial)))
        return cross_val_score(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=1).mean()

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        study_name=name,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study


def tune_all_models(X, y, cv) -> dict[str, optuna.Study]:
    studies = {}
    for name in MODEL_REGISTRY:
        print(f"\n▶ {name} 튜닝 중 ({N_TRIALS} trials, 기준: {TUNING_METRIC})")
        studies[name] = tune_model(name, X, y, cv)
    return studies


def summarize_studies(studies: dict[str, optuna.Study]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "Model": name,
            f"Best CV {TUNING_METRIC}": round(study.best_value, 4),
            "Best Params": study.best_params,
        }
        for name, study in studies.items()
    ]).sort_values(f"Best CV {TUNING_METRIC}", ascending=False, ignore_index=True)


# ---------------------------------------------------------------------
# 실행
# ---------------------------------------------------------------------
if __name__ == "__main__":
    X, y = load_and_clean()
    print(summarize(X))

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    # (1) Baseline: 기본 파라미터로 6개 모델 비교
    baseline_df = evaluate_models({n: make_model(n) for n in MODEL_REGISTRY}, X, y, cv)
    print("\n[Baseline 5-Fold 성능]")
    print(baseline_df.to_string(index=False))

    # (2) Optuna 튜닝: 6개 모델 전부
    studies = tune_all_models(X, y, cv)

    # (3) 모델별 best 하이퍼파라미터 확인
    print("\n" + "=" * 60)
    for name, study in studies.items():
        print(f"[{name}] best {TUNING_METRIC}: {study.best_value:.4f}")
        print(f"    best params: {study.best_params}")
    print("=" * 60)

    best_summary = summarize_studies(studies)
    print(best_summary.to_string(index=False))

    # (4) best 파라미터로 재평가 (baseline과 동일한 지표로 비교)
    tuned_models = {f"{n} (Tuned)": make_model(n, s.best_params) for n, s in studies.items()}
    tuned_df = evaluate_models(tuned_models, X, y, cv)
    print("\n[Tuned 5-Fold 성능]")
    print(tuned_df.to_string(index=False))

     Column    Dtype  N_Unique  Null_Count  Null_Ratio(%)  \
0    Pclass    int64         3           0           0.00   
1       Sex   object         2           0           0.00   
2       Age  float64        88         177          19.87   
3     SibSp    int64         7           0           0.00   
4     Parch    int64         7           0           0.00   
5      Fare  float64       248           0           0.00   
6  Embarked   object         3           2           0.22   
7     Title   object         5           0           0.00   

                              Samples  
0                           [3, 1, 2]  
1                      [male, female]  
2      [22.0, 38.0, 26.0, 35.0, 54.0]  
3                     [1, 0, 3, 4, 2]  
4                     [0, 1, 2, 5, 3]  
5  [7.25, 71.2833, 7.925, 53.1, 8.05]  
6                           [S, C, Q]  
7       [Mr, Mrs, Miss, Master, Rare]  

[Baseline 5-Fold 성능]
              Model  FN+FP (오분류 수)  FPR (오탐률)  Recall (재현율)  Precisi

  0%|          | 0/50 [00:00<?, ?it/s]


▶ Logistic Regression 튜닝 중 (50 trials, 기준: f1)


  0%|          | 0/50 [00:00<?, ?it/s]


▶ Naive Bayes 튜닝 중 (50 trials, 기준: f1)


...


▶ Random Forest 튜닝 중 (50 trials, 기준: f1)


  0%|          | 0/50 [00:00<?, ?it/s]


▶ XGBoost 튜닝 중 (50 trials, 기준: f1)


  0%|          | 0/50 [00:00<?, ?it/s]


[KNN] best f1: 0.7662
    best params: {'n_neighbors': 17, 'weights': 'uniform', 'p': 1}
[Logistic Regression] best f1: 0.7767
    best params: {'C': 6.846801912794278, 'penalty': 'l2'}
[Naive Bayes] best f1: 0.7087
    best params: {'var_smoothing': 1.767016940294792e-09}
[Decision Tree] best f1: 0.7710
    best params: {'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 15, 'min_samples_leaf': 5}
[Random Forest] best f1: 0.7853
    best params: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2'}
[XGBoost] best f1: 0.7957
    best params: {'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.06488476245157623, 'subsample': 0.8225675791992874, 'colsample_bytree': 0.7648434506455819, 'min_child_weight': 2, 'gamma': 0.15668151227510557}
              Model  Best CV f1                                                                                                                                                           